In [31]:
# %% Minimal setup from class

import os, json, textwrap, re, time
import requests

API_KEY  = os.getenv("OPENAI_API_KEY", "cse476")
API_BASE = os.getenv("API_BASE", "http://10.4.58.53:41701/v1")  
MODEL    = os.getenv("MODEL_NAME", "bens_model")              

SYSTEM_PROMPT = "You are a helpful assistant with limited text output. Reply with only the final answer—no explanation. There is no need to repeat the question."
TEMPERATURE   = 0.25 #Must be a float

def call_model_chat_completions(prompt: str,
                                system: str = SYSTEM_PROMPT,
                                model: str = MODEL,
                                temperature: float = TEMPERATURE,
                                timeout: int = 60) -> dict:
    """
    Calls an OpenAI-style /v1/chat/completions endpoint and returns:
    { 'ok': bool, 'text': str or None, 'raw': dict or None, 'status': int, 'error': str or None, 'headers': dict }
    """
    url = f"{API_BASE}/chat/completions"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type":  "application/json",
    }
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt}
        ],
        "temperature": temperature,
        "max_tokens": 350,
    }

    try:
        resp = requests.post(url, headers=headers, json=payload, timeout=timeout)
        status = resp.status_code
        hdrs   = dict(resp.headers)
        if status == 200:
            data = resp.json()
            text = data.get("choices", [{}])[0].get("message", {}).get("content", "")
            return {"ok": True, "text": text, "raw": data, "status": status, "error": None, "headers": hdrs}
        else:
            # try best-effort to surface error text
            err_text = None
            try:
                err_text = resp.json()
            except Exception:
                err_text = resp.text
            return {"ok": False, "text": None, "raw": None, "status": status, "error": str(err_text), "headers": hdrs}
    except requests.RequestException as e:
        return {"ok": False, "text": None, "raw": None, "status": -1, "error": str(e), "headers": {}}

def self_evaluate(question, prediction, expected_answer, model=MODEL):
    """
    Use the model itself as a strict grader.
    Returns True if the model says the prediction matches the expected answer; else False.
    Falls back to a simple normalized string compare if the model's reply is malformed.
    """
    import re

    system = "You are a strict grader. Reply with exactly True or False. No punctuation. No explanation."
    prompt = f"""You are grading a question-answer pair.

Return exactly True if the PREDICTION would be accepted as correct for the EXPECTED_ANSWER.
Otherwise, return False.

QUESTION:
{question}

PREDICTION:
{prediction}

EXPECTED_ANSWER:
{expected_answer}

Answer with exactly: True or False
"""

    r = call_model_chat_completions(
        prompt,
        system=system,
        model=model,
        temperature=0.0,
    )

    reply = (r.get("text") or "").strip().lower()
    if reply.startswith("true"):
        return True
    if reply.startswith("false"):
        return False

    # Fallback: simple normalization-based equality
    norm = lambda s: re.sub(r"\s+", " ", (s or "").strip().lower())
    return norm(prediction) == norm(expected_answer)

def self_evaluate_tests(tests, model=MODEL, grader_model=None, sleep_sec=0.2, verbose=True):
    """
    Run the tests by querying the model for each prompt, then use LLM-as-a-judge
    (self_evaluate) to determine correctness.

    Args:
        tests: list of dicts with keys: id, prompt, expected (and optionally type)
        model: model used to generate predictions
        grader_model: model used to judge correctness (defaults to `model` if None)
        sleep_sec: small delay between calls to be polite to the API
        verbose: if True, print a summary line per test

    Returns:
        rows: list of dicts with fields:
              id, expected, got, correct, status, error
    """
    import time

    judge_model = grader_model or model
    rows = []

    for t in tests:
        #1) Get model prediction
        if t.get("input"):
            r = call_model_chat_completions(
                t["input"],
                system="You are a careful solver. Reply ONLY with the final answer, nothing else.",
                model=model,
                temperature=TEMPERATURE,
            )
            #got = (r.get("text") or "").strip()
            got = agent_loop(t["input"])
            
            # 2) LLM-as-a-judge: strict True/False
            is_correct = self_evaluate(
                question=t["input"],
                prediction=got,
                expected_answer=t["output"],
                model=judge_model,
            )
        else:
            r = call_model_chat_completions(
                t["prompt"],
                system="You are a careful solver. Reply ONLY with the final answer, nothing else.",
                model=model,
                temperature=TEMPERATURE,
            )        
            #got = (r.get("text") or "").strip()    
            got = agent_loop(t["prompt"])

            # 2) LLM-as-a-judge: strict True/False
            is_correct = self_evaluate(
                question=t["prompt"],
                prediction=got,
                expected_answer=t["expected"],
                model=judge_model,
            )



        row = {
            "id": t.get("id", "<unnamed>"),
            "output": t["output"],
            "got": got,
            "correct": bool(is_correct),
            "status": r.get("status"),
            "error": r.get("error"),
        }
        rows.append(row)

        if verbose:
            mark = "✅" if is_correct else "❌"
            print(f"{mark} {row['id']}: output={row['output']!r}, got={row['got']!r} (HTTP {row['status']})")
            if row["error"]:
                print("   error:", row["error"])

        if sleep_sec:
            time.sleep(sleep_sec)

    return rows


In [32]:
def reasoning_via_planning(prior:str, question: str,) -> dict:
    prior_reasoning = "\nPrior Reasoning: " + prior + "\n\n"
    reasoning_str = "Create a plan using as little words as possible. Using prior reasoning, decompose the problem into a few step to solve the question provided. Execute each step in order to arrive at the final answer.If the question involves math, write a python program that solves the question and exports an answer. Make sure your answer solves the question provided.\n\n Question: "

    r = call_model_chat_completions(
            reasoning_str + question + prior_reasoning,
            system="You are a planner. Provide a short, structured step-by-step plan to solve the question.",
            model=MODEL,
            temperature=0.35,
        )
    got = (r.get("text") or "").strip()
    return got

In [33]:
def tree_of_thought(question: str, n_paths: int, prior: str = None,):
    tot_str = "You will decompose the problem down into {n_paths} distinct possible solution paths to solve the question provided. Depending on the problem, write a short reasonable path that is different from every other path created but still leads to the answer. Expected output should be in the form of: path1:<>, \npath2:<>,etc.\n\n Question: "
    r = call_model_chat_completions(
            tot_str.format(n_paths=n_paths) + question,
            system="You are a problem solver that creates multiple distinct solution paths to solve a problem. Use as little words as possible.",
            model=MODEL,
            temperature=TEMPERATURE,
        )
    #Divide into n_paths
    raw = r.get("text") or ""
    thoughts = []
    for n in range(1, n_paths + 1):
        path = raw.find(f"path{n}:")
        end = raw.find(f"path{n+1}:")
        if end == -1:
            end = len(raw)
        thoughts.append(raw[path:end].strip())
    return thoughts

In [34]:
def self_consistency(question: str, n_paths: int, prior: str = "",):
    paths = []
    for n in range(1, n_paths + 1):
        r = call_model_chat_completions(
                prior + question,
                system="",
                model=MODEL,
                temperature=TEMPERATURE,
            )
        got = (r.get("text") or "").strip()
        paths.append(got)
    return paths

In [35]:
def double_check(prior:str, question: str):
    original_question = "Original Question: " + question + "\n"
    check_str = "Verify this solution.If incorrect, output the correct solution. Output ONLY the final answer in the exact required format. No explanations, no markdown, no labels."
    r = call_model_chat_completions(
            original_question + check_str + prior,
            system="You verify solutions and output ONLY the final answer. No 'Answer:', no bold **, no quotes, no extra text. Just the raw answer value.",
            model=MODEL,
            temperature=TEMPERATURE,
        )
    got = (r.get("text") or "").strip()
    return got

In [36]:
import requests, trafilatura
from urllib.parse import quote
# Returns text content of most relevant Wikipedia page for a question
def wiki_tool(question: str):
    p = '''Given a question, provide the title of the most relevant Wikipedia page that answers the question. 
            Only provide the title, no explanations.
            Examples: 
            Question: Which genus of moth in the world's seventh-largest country contains only one species?
            Answer: Crambidae
            Question: What U.S Highway gives access to Zilpo Road, and is also known as Midland Trail?
            Answer: US 60
            \n\n Question: '''
    r = call_model_chat_completions(
            p + question,
            system="You are a Wikipedia search assistant. Given a question, return the title of the most relevant Wikipedia page that answers the question. Reply with only the title, no explanations.",
            model=MODEL,
            temperature=0.3,
        )
    title = (r.get("text") or "").strip()
    
    try:
        #Calls a wiki API to get page of a topic
        url = f"https://en.wikipedia.org/api/rest_v1/page/mobile-html/{quote(title)}"
        headers = {"User-Agent": "MilkBot/1.0 (https://github.com/Isaiah-Milkey)", "Accept": "text/html"}
        r = requests.get(url, headers=headers, timeout=20)
        r.raise_for_status()

        extracted = trafilatura.extract(
            r.text,
            include_tables=True,
            include_comments=False,
            output_format="txt"  
        )
    except Exception as e:
        print(f"Error fetching Wikipedia page for title '{title}': {e}")
        extracted = "No extractable content found."
    return extracted or "No Wikipedia content found."

In [37]:
#Following example from: https://github.com/paaxel/llama-starter-examples/blob/main/6-hello-llama-rag.ipynb
from langchain.docstore.document import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
import json

data = []
with open("cse476_final_project_dev_data.json", "r") as tests:
    DEV_DATA = json.load(tests)

data = []
for test in DEV_DATA:
    data.append(
        Document(
            page_content = str(test["output"]),
            metadata = {"input": test["input"], "Domain": test["domain"]}
        )
    )

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = HuggingFaceEmbeddings(model_name=embedding_model_name)
vector_store = FAISS.from_documents(data, embedding_model)

def RAG_tool(question: str, num_examples: int = 1):
    query = vector_store.similarity_search(question, k=num_examples)
    
    content = ""
    for i, doc in enumerate(query, 1):
        content += f"Example {i}:\n"
        content += f"Input: {doc.metadata['input']}\n"
        content += f"Output: {doc.page_content}\n\n"
    
    return content.strip()


In [ ]:
def calc_tool(input_question: str):
    retrieved_examples = RAG_tool(input_question, num_examples=2)


    MATH_AGENT_PROMPT = f"""
    You MUST respond in exactly one of these two formats: 
    Arithmetic: <arithmetic expression>
    FINAL: <answer>  

    - use only numbers, + - * / **, parentheses, and round(x, ndigits)
    - avoid using parentheses withing parentheses where possible AND always close open parentheses. eg: (((2*3)*4)*5)+1
    Example of Arithmetic: round((3*2.49)*1.07, 2)
    Example of FINAL: FINAL: 23
    Return ONE line with NO explanations. No other text.
    Examples from similar problems:
    {retrieved_examples}
    """
    r = call_model_chat_completions(
            prompt= f"""Question: {input_question}
            If you need arithmetic to solve the question, reply as:
            <expression>
            
            Otherwise reply:
            FINAL: <answer> """,
            system=MATH_AGENT_PROMPT,
            model=MODEL,
            temperature=0,
        )

    case = (r.get("text") or "").strip().lower()
    
    if "arithmetic:" in case:
        try:
            expression = case.lstrip("arithmetic:")
            if expression.count("(") > expression.count(")"):
                expression += ")"
            elif expression.count("(") > expression.count(")"):
                expression = "(" + expression
            return eval(expression)
        except:
            return case
    else:
        return case

In [ ]:
# Case definitions for each case: math, coding, future_prediction, planning, and common_sense

#Code inspired by Mini Lab 5 and this youtube video: https://www.youtube.com/watch?v=Uz7pWszyi6k
def case_math(input_question: str):
    #print("entered case_math function")
    RAG_content = RAG_tool(input_question, num_examples=2)
    prior = RAG_content + "\nSolve the following question:\n" + input_question
    curr = reasoning_via_planning("For solving math problems, I can create a simple python program that solves the question- and outputs the answer.", prior)
    curr = calc_tool(input_question+curr)
    ans = double_check(str(curr), input_question)
    return ans

def case_coding(input_question: str):
    RAG_content = RAG_tool(input_question, num_examples=2)
    prior = RAG_content + "\nSolve the following question:\n" + input_question
    curr = reasoning_via_planning("For solving coding problems, I can create a simple python program that solves the question- and outputs the answer.", prior)
    curr = double_check(curr, input_question) #Get an output, then perform RAG search on the solution and repeat

    RAG_content = RAG_tool(curr, num_examples=5)
    prior = RAG_content + "\nSolve the following question:\n" + input_question
    curr = reasoning_via_planning("For solving coding problems, I can create a simple python program that solves the question- and outputs the answer.", prior)
    r = call_model_chat_completions(
            prompt= f"""Solve this provided problem by creating a python script using the provided reference.
            PROBLEM: {input_question}
            REFERENCE: {curr}
            """,
            system="""You are a coding agent that writes simple and reliable code that will always execute correctly.
            You MUST respond using this format:
            Python script: <answer> 
            
            Example: x_values = np.linspace(0, 2 * np.pi, 400)\n    fig, axs = plt.subplots(2)\n
            Example: combined_matrix = np.concatenate((matrix1, matrix2), axis=1)\n    df = pd.DataFrame(combined_matrix)\n    return df.to_string(index=False, header=False)""",
            model=MODEL,
            temperature=0.2,
        )
    final = (r.get("text") or "").strip()
    return final

    

In [ ]:
# Future:
# Good vid for reference: https://www.youtube.com/watch?v=ahnGLM-RC1Y
# Alter RAG: Use cosine similarity? And have RAG call based on an LLM assumption of the answer: then perform RAG search on that.
#   Increase Number of examples
# Add individual functions for all cases
# Have multiple wiki calls if needed/ cant find title

def agent_loop(input_question: str):
    #Have the model decide on a strategy to solve the problem
    r = call_model_chat_completions(
            "Decide what type of category this question falls into: math, coding, general_knowledge, reasoning, or other. Respond with only the category name." + "\n\n Question: " + input_question,
            system="You are a helpful assistant that classifies questions into categories.",
            model=MODEL,
            temperature=0,
        )
    case = (r.get("text") or "").strip().lower()
    case = case.split()[0] if case.split() else "other"
    print(f"Case decided: {case}\n")
    if case == "math":
        #Math/Coding solving strategy
        print("Using math solving strategy\n")
        return case_math(input_question)
    
    elif case == "coding":
        #Coding solving strategy
        print("Using coding solving strategy\n")
        return case_coding(input_question)
    
    elif case == "general_knowledge":
        #Use wiki tool
        print("Using general knowledge solving strategy with wiki tool\n")
        RAG_content = RAG_tool(input_question, num_examples=2)
        prior = RAG_content
        curr = prior + wiki_tool(question=input_question)
        ans = double_check(curr, input_question)
        if ans != "":
            return ans
        else:
            paths = self_consistency(prior, n_paths=2)
            combined = "These are the different solution paths: \n"
            for p in paths:
                combined += p + "\n"
            ans = double_check(combined, input_question)
            return ans
    elif case == "reasoning":
        #Use reasoning via planning
        print("Using reasoning via planning solving strategy\n")
        RAG_content = RAG_tool(input_question, num_examples=2)
        prior = RAG_content
        curr = reasoning_via_planning(prior, input_question)
        return curr
    elif case == "other":
        #Fallback to self consistency
        print("Using other case: self consistency solving strategy\n")
        RAG_content = RAG_tool(input_question, num_examples=3)
        prior = RAG_content + "\nSolve the following question:\n" + input_question
        paths = self_consistency(prior, n_paths=2)
        combined = "These are the different solution paths: \n"
        for p in paths:
            combined += p + "\n"
        ans = double_check(combined, input_question)
        if ans != "":
            return ans
        else:
            return agent_loop(input_question)
    else:
        print("Did not catch a case: Using self consistency\n")
        RAG_content = RAG_tool(input_question, num_examples=3)
        prior = RAG_content + "\nSolve the following question:\n" + input_question
        paths = self_consistency(prior, n_paths=2)
        combined = "These are the different solution paths: \n"
        for p in paths:
            combined += p + "\n"
        ans = double_check(combined, input_question)
        if ans != "":
            return ans
        else:
            return agent_loop(input_question)
    

In [50]:
import json
import random

with open("cse476_final_project_dev_data.json", "r") as tests:
    DEV_DATA = json.load(tests)

#Get test batches by domain/random
def filter_domain(domain: str):
    filtered = []
    for test in DEV_DATA:
        if test.get("domain") == domain:
            filtered.append(test)
    return filtered

def get_batch(num: int, domain: str = None, is_random: bool = False):
    #random.seed(315)
    if domain:
        data = filter_domain(domain)
    else:
        data = DEV_DATA

    if is_random:
        return random.sample(data, num)
    else:
        return data[:num]

In [64]:

# Example:
# test = reasoning_via_planning("", question="A farmer has 17 sheep and all but 9 are lost. How many sheep are left on the farm?")

# print(test)
#print(test2)

tests = get_batch(10, domain="coding", is_random=True)
self_evaluate_tests(tests, model=MODEL, sleep_sec=0.5, verbose=True)


Case decided: coding

Using coding solving strategy

✅ <unnamed>: output='    df = pd.read_csv(csv_file_path)\n    groupby_data = df.groupby(col1_name)[col2_name].mean()\n\n    _, ax = plt.subplots(figsize=(10, 6))\n    ax.bar(groupby_data.index, groupby_data.values)\n    ax.set_title(f"Mean of {col2_name} Grouped by {col1_name}")\n    ax.set_xlabel(col1_name)\n    ax.set_ylabel(f"Mean of {col2_name}")\n\n    return ax', got='Python script: \n```python\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\ndef task_func(csv_file_path, col1_name="column1", col2_name="column2"):\n    df = pd.read_csv(csv_file_path)\n    grouped = df.groupby(col1_name)[col2_name].mean()\n    fig, ax = plt.subplots()\n    ax.bar(grouped.index, grouped.values)\n    ax.set_title(f"Mean of {col2_name} Grouped by {col1_name}")\n    ax.set_xlabel(col1_name)\n    ax.set_ylabel(f"Mean of {col2_name}")\n    return ax\n```' (HTTP 200)
Case decided: coding

Using coding solving strategy

✅ <unnamed>: output="    X

[{'id': '<unnamed>',
  'output': '    df = pd.read_csv(csv_file_path)\n    groupby_data = df.groupby(col1_name)[col2_name].mean()\n\n    _, ax = plt.subplots(figsize=(10, 6))\n    ax.bar(groupby_data.index, groupby_data.values)\n    ax.set_title(f"Mean of {col2_name} Grouped by {col1_name}")\n    ax.set_xlabel(col1_name)\n    ax.set_ylabel(f"Mean of {col2_name}")\n\n    return ax',
  'got': 'Python script: \n```python\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\ndef task_func(csv_file_path, col1_name="column1", col2_name="column2"):\n    df = pd.read_csv(csv_file_path)\n    grouped = df.groupby(col1_name)[col2_name].mean()\n    fig, ax = plt.subplots()\n    ax.bar(grouped.index, grouped.values)\n    ax.set_title(f"Mean of {col2_name} Grouped by {col1_name}")\n    ax.set_xlabel(col1_name)\n    ax.set_ylabel(f"Mean of {col2_name}")\n    return ax\n```',
  'correct': True,
  'status': 200,
  'error': None},
 {'id': '<unnamed>',
  'output': "    X = np.linspace(-10, 10, 400)  #

In [ ]:
#!/usr/bin/env python3
"""
Generate a placeholder answer file that matches the expected auto-grader format.

Replace the placeholder logic inside `build_answers()` with your own agent loop
before submitting so the ``output`` fields contain your real predictions.

Reads the input questions from cse_476_final_project_test_data.json and writes
an answers JSON file where each entry contains a string under the "output" key.
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, List


INPUT_PATH = Path("cse_476_final_project_test_data.json")
OUTPUT_PATH = Path("cse_476_final_project_answers.json")


def load_questions(path: Path) -> List[Dict[str, Any]]:
    with path.open("r", encoding="utf-8") as fp:
        data = json.load(fp)
    if not isinstance(data, list):
        raise ValueError("Input file must contain a list of question objects.")
    return data


def build_answers(questions: List[Dict[str, Any]]) -> List[Dict[str, str]]:
    answers = []
    for idx, question in enumerate(questions, start=1):
        # Example: ass0ume you have an agent loop that produces an answer string.
        # real_answer = agent_loop(question["input"])
        # answers.append({"output": real_answer})
        answer = agent_loop(question["input"])
        answers.append({"output": answer})
    return answers

def validate_results(
    questions: List[Dict[str, Any]], answers: List[Dict[str, Any]]
) -> None:
    if len(questions) != len(answers):
        raise ValueError(
            f"Mismatched lengths: {len(questions)} questions vs {len(answers)} answers."
        )
    for idx, answer in enumerate(answers):
        if "output" not in answer:
            raise ValueError(f"Missing 'output' field for answer index {idx}.")
        if not isinstance(answer["output"], str):
            raise TypeError(
                f"Answer at index {idx} has non-string output: {type(answer['output'])}"
            )
        if len(answer["output"]) >= 5000:
            raise ValueError(
                f"Answer at index {idx} exceeds 5000 characters "
                f"({len(answer['output'])} chars). Please make sure your answer does not include any intermediate results."
            )


#questions = load_questions(INPUT_PATH)
#answers = build_answers(questions)



In [ ]:
with OUTPUT_PATH.open("w", encoding="utf-8") as fp:
    json.dump(answers, fp, ensure_ascii=False, indent=2)

with OUTPUT_PATH.open("r", encoding="utf-8") as fp:
    saved_answers = json.load(fp)
validate_results(questions, saved_answers)
print(
    f"Wrote {len(answers)} answers to {OUTPUT_PATH} "
    "and validated format successfully."
)